# 🧠 LSTM Short Text Classification — Google Colab Training

> **Đề tài**: Xây dựng mô hình học sâu LSTM cho bài toán phân loại văn bản ngắn tiếng Việt

### Trước khi chạy
1. **Bật GPU**: `Runtime → Change runtime type → T4 GPU → Save`
2. **Chạy tuần tự** từ trên xuống (hoặc `Ctrl+F9` để chạy tất cả)
3. Kết quả tự động lưu vào **Google Drive** → không mất khi session hết giờ

### Thời gian ước tính (T4 GPU)
| Bước | Thời gian |
|------|-----------|
| Setup + Preprocessing | ~3 phút |
| Train BiLSTM ★ | ~5–8 phút |
| Train DNN | ~2 phút |
| Train XLM-R | ~15–20 phút |
| Ablation study | ~12 phút |
| Evaluate + Figures | ~3 phút |
| **Tổng** | **~45–50 phút** |

## ☁️ PHẦN 1 — Setup

In [ ]:
# ── 1.1 Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted')

In [ ]:
# ── 1.2 Clone repo từ GitHub ──────────────────────────────────────────────────
import os, shutil
from pathlib import Path

REPO_URL  = 'https://github.com/nhiney/lstm-short-text-classification.git'
WORK_DIR  = Path('/content/lstm-short-text-classification')
DRIVE_DIR = Path('/content/drive/MyDrive/lstm-text-classification')

DRIVE_DIR.mkdir(parents=True, exist_ok=True)

if WORK_DIR.exists():
    print('✓ Repo already cloned')
else:
    print(f'Cloning {REPO_URL} ...')
    !git clone "{REPO_URL}" "{WORK_DIR}"
    print('✓ Clone done')

os.chdir(WORK_DIR)
print(f'✓ Working dir: {os.getcwd()}')
!ls

In [ ]:
# ── 1.3 Cài thêm thư viện (Colab đã có torch/pandas/sklearn) ─────────────────
import subprocess, sys

packages = [
    'transformers>=4.44.0',
    'pyyaml>=6.0',
    'openpyxl>=3.1.0',
    'seaborn>=0.13.0',
    'emoji>=2.12.0',
    'accelerate>=0.33.0',
]
print('Installing extra packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)
print('✓ Done')

In [ ]:
# ── 1.4 Kiểm tra môi trường ───────────────────────────────────────────────────
import sys
sys.path.insert(0, str(WORK_DIR))

import torch
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU     : {gpu}  ({vram:.1f} GB)')
else:
    print('GPU     : NOT AVAILABLE — hãy bật T4 GPU!')

from src.utils.config import CFG, get_device
DEVICE = get_device()
print(f'Device  : {DEVICE}')
print(f'Labels  : {CFG.labels.codes}')
print('\n✅ Environment OK')

In [ ]:
# ── 1.5 Hàm tiện ích: backup / restore từ Drive ───────────────────────────────
import shutil

def save_to_drive():
    """Lưu outputs/ và data/processed/ lên Drive để không mất khi session hết."""
    pairs = [
        (WORK_DIR / 'data' / 'processed', DRIVE_DIR / 'data_processed'),
        (WORK_DIR / 'outputs',            DRIVE_DIR / 'outputs'),
    ]
    for src, dst in pairs:
        if src.exists():
            if dst.exists(): shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f'  ✓ Drive ← {src.name}')

def restore_from_drive():
    """Khôi phục outputs/ và processed data từ Drive (session mới)."""
    pairs = [
        (DRIVE_DIR / 'data_processed', WORK_DIR / 'data' / 'processed'),
        (DRIVE_DIR / 'outputs',        WORK_DIR / 'outputs'),
    ]
    for src, dst in pairs:
        if src.exists():
            if dst.exists(): shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f'  ✓ Restored: {dst.name}')
        else:
            print(f'  (not on Drive yet: {src.name})')

# Khôi phục checkpoint nếu có session trước
print('Checking Drive for previous checkpoints...')
restore_from_drive()

## 🔧 PHẦN 2 — Preprocessing

In [ ]:
from pathlib import Path

vocab_path = Path('data/processed/vocabulary.pkl')

if vocab_path.exists():
    from src.preprocessing.vocabulary import Vocabulary
    vocab = Vocabulary.load(vocab_path)
    print(f'✓ Vocabulary đã có sẵn  (size={vocab.size:,})')
else:
    print('Chạy preprocessing...')
    from src.preprocessing.preprocess import run_preprocessing
    out = run_preprocessing()
    vocab = out['vocabulary']
    print(f'\n✓ Preprocessing xong!')
    print(f'  Train={len(out["train_df"]):,} | Val={len(out["val_df"]):,} | Test={len(out["test_df"]):,}')
    print(f'  Vocabulary size: {vocab.size:,}')
    save_to_drive()

## 🚀 PHẦN 3 — Training

In [ ]:
# ── 3.1 BiLSTM + Attention (mô hình chính của đề tài) ★ ───────────────────────
lstm_ckpt = Path('outputs/models/lstm_best.pt')

if lstm_ckpt.exists():
    import json
    h = json.load(open('outputs/logs/lstm_history.json'))
    print(f'✓ lstm_best.pt đã có — epochs={len(h["train_loss"])} | test_acc={h.get("test_acc","N/A")}')
else:
    print('Training BiLSTM + Attention...')
    from src.training.train_lstm import train_lstm
    h = train_lstm()
    print(f'\n✅ BiLSTM done!  acc={h["test_acc"]:.4f}  F1={h["test_f1"]:.4f}  ({h["training_time_s"]:.0f}s)')
    save_to_drive()

In [ ]:
# ── 3.2 DNN + TF-IDF (baseline) ───────────────────────────────────────────────
dnn_ckpt = Path('outputs/models/dnn_best.pt')

if dnn_ckpt.exists():
    import json
    h = json.load(open('outputs/logs/dnn_history.json'))
    print(f'✓ dnn_best.pt đã có — test_acc={h.get("test_acc","N/A")}')
else:
    print('Training DNN + TF-IDF...')
    from src.training.train_dnn import train_dnn
    h = train_dnn()
    print(f'✅ DNN done!  acc={h["test_acc"]:.4f}')
    save_to_drive()

In [ ]:
# ── 3.3 XLM-RoBERTa (comparison / upper-bound) — ~20 phút với T4 ──────────────
xlmr_ckpt = Path('outputs/models/xlmr_best.pt')

if xlmr_ckpt.exists():
    import json
    h = json.load(open('outputs/logs/xlmr_history.json'))
    print(f'✓ xlmr_best.pt đã có — test_acc={h.get("test_acc","N/A")}')
else:
    print('Training XLM-RoBERTa (~20 phút, cần T4)...')
    from src.training.train_xlmr import train_xlmr
    h = train_xlmr()
    print(f'✅ XLM-R done!  acc={h["test_acc"]:.4f}')
    save_to_drive()

## 🔬 PHẦN 4 — Ablation Study

So sánh 3 biến thể để chứng minh đóng góp của từng component:
- **LSTM** — unidirectional, không attention
- **BiLSTM** — bidirectional, không attention  
- **BiLSTM + Attention** ★ — mô hình đề xuất đầy đủ

In [ ]:
abl_path = Path('outputs/logs/ablation_results.json')

if abl_path.exists():
    import json
    abl = json.load(open(abl_path))
    print('✓ Ablation results đã có:')
    print(f'\n  {"Variant":<38} {"Acc":>7} {"F1":>7}')
    print('  ' + '─'*55)
    for name, m in abl.items():
        tag = ' ★' if 'Attention' in name else ''
        print(f'  {name+tag:<40} {m["accuracy"]:>7.4f} {m["f1"]:>7.4f}')
else:
    print('Running ablation study (~12 phút)...')
    from src.training.train_ablation import run_ablation
    res = run_ablation()
    print('\n✅ Ablation done!')
    print(f'\n  {"Variant":<38} {"Acc":>7} {"F1":>7} {"Params":>12}')
    print('  ' + '─'*65)
    for name, m in res.items():
        tag = ' ★' if 'Attention' in name else ''
        print(f'  {name+tag:<40} {m["accuracy"]:>7.4f} {m["f1"]:>7.4f} {m["num_params"]:>12,}')
    save_to_drive()

## 📊 PHẦN 5 — Evaluation & Visualization

In [ ]:
# Đánh giá tất cả models
from src.evaluation.evaluate import run_evaluation
import pandas as pd

results = run_evaluation()

rows = [{
    'Model': name,
    'Accuracy':  f"{m['accuracy']:.4f}",
    'Precision': f"{m['precision']:.4f}",
    'Recall':    f"{m['recall']:.4f}",
    'F1':        f"{m['f1']:.4f}",
    'Params':    f"{m['num_params']:,}",
    'ms/sample': f"{m['inference_ms_per_sample']:.2f}",
} for name, m in results.items()]

print('\n' + '='*72)
print('  MODEL COMPARISON')
print('='*72)
print(pd.DataFrame(rows).to_string(index=False))
save_to_drive()

In [ ]:
# Tạo tất cả figures
import matplotlib
matplotlib.use('Agg')
from src.evaluation.visualization import generate_all_figures

generate_all_figures()

figs = sorted(Path('outputs/figures').glob('*.png'))
print(f'✅ {len(figs)} figures created:')
for f in figs:
    print(f'   {f.name}')
save_to_drive()

In [ ]:
# Hiển thị tất cả figures
from IPython.display import Image, display
from pathlib import Path

for fpath in sorted(Path('outputs/figures').glob('*.png')):
    print(f'\n── {fpath.stem} ──')
    display(Image(str(fpath), width=900))

## 🔍 PHẦN 6 — Error Analysis & Attention Visualization

In [ ]:
# Error analysis cho BiLSTM
from src.evaluation.error_analysis import analyze_errors, print_error_summary

report = analyze_errors('lstm', top_n=20)
print_error_summary(report)
save_to_drive()

In [ ]:
# Attention weights visualization
import matplotlib.pyplot as plt
import numpy as np
from src.inference.predict import LSTMPredictor

predictor = LSTMPredictor()

test_cases = [
    ('Hôm nay tôi rất vui được gặp mọi người!!!', 'JOY'),
    ('Tôi buồn lắm, không muốn làm gì cả... huhu', 'SAD'),
    ('Tức quá! Sao có thể làm vậy được chứ???',   'ANG'),
    ('Ôi trời ơi, không ngờ lại như vậy!',         'SUR'),
]

fig, axes = plt.subplots(len(test_cases), 1, figsize=(13, 3.2 * len(test_cases)))

for ax, (text, true_lbl) in zip(axes, test_cases):
    result  = predictor.predict(text)
    weights = predictor.get_attention_weights(text)

    tokens = list(weights.keys())
    scores = np.array(list(weights.values()))
    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

    bars = ax.barh(range(len(tokens)), scores,
                   color=plt.cm.Oranges(scores), edgecolor='white')
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Attention weight')
    ok = '✓' if result['predicted_label'] == true_lbl else '✗'
    ax.set_title(
        f'{ok} Pred: {result["predicted_name"]} ({result["confidence"]:.0%})  |  True: {true_lbl}\n"{text}"',
        fontsize=10
    )
    ax.grid(axis='x', alpha=0.3)

fig.suptitle('Attention Weight Visualization — BiLSTM+Attention', fontweight='bold', y=1.01)
plt.tight_layout()
out_path = 'outputs/figures/attention_visualization.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved: {out_path}')
save_to_drive()

## 📝 PHẦN 7 — Tóm tắt kết quả

In [ ]:
import json
from pathlib import Path

SEP = '='*72
print(SEP)
print('  KẾT QUẢ THỰC NGHIỆM — LSTM Short Text Classification')
print(SEP)

# Main comparison
comp_path = Path('outputs/reports/comparison.json')
if comp_path.exists():
    comp = json.load(open(comp_path))
    print(f'\n  {"Model":<22} {"Acc":>7} {"Prec":>7} {"Recall":>7} {"F1":>7} {"Params":>12}')
    print('  ' + '─'*65)
    for name, m in comp.items():
        tag = ' ★' if name == 'BiLSTM' else ''
        print(f'  {name+tag:<24} {m["accuracy"]:>7.4f} {m["precision"]:>7.4f} '
              f'{m["recall"]:>7.4f} {m["f1"]:>7.4f} {m["num_params"]:>12,}')

# Ablation
abl_path = Path('outputs/logs/ablation_results.json')
if abl_path.exists():
    abl = json.load(open(abl_path))
    print(f'\n  ABLATION STUDY')
    print(f'  {"Variant":<42} {"Acc":>7} {"F1":>7}')
    print('  ' + '─'*58)
    for name, m in abl.items():
        tag = ' ★' if 'Attention' in name else ''
        print(f'  {name+tag:<44} {m["accuracy"]:>7.4f} {m["f1"]:>7.4f}')

# Per-class F1 của BiLSTM
if comp_path.exists() and 'BiLSTM' in json.load(open(comp_path)):
    pcf1 = json.load(open(comp_path))['BiLSTM'].get('per_class_f1', {})
    if pcf1:
        print('\n  Per-class F1 — BiLSTM+Attention:')
        for cls, f1 in sorted(pcf1.items(), key=lambda x: -x[1]):
            bar = '█' * int(f1 * 28)
            print(f'    {cls}  {bar:<30} {f1:.4f}')

print('\n' + SEP)
print(f'  Figures : outputs/figures/')
print(f'  Report  : outputs/reports/comparison.json')
print(f'  Drive   : {DRIVE_DIR}')

In [ ]:
# Demo predict
from src.inference.predict import LSTMPredictor
predictor = LSTMPredictor()

samples = [
    'Hôm nay vui lắm, được điểm cao nè!!!',
    'Chán quá không biết phải làm sao...',
    'Tức chết đi được! Sao lại như vậy???',
    'Ôi không ngờ lại được quà như vậy!',
    'Bình thường thôi, không có gì đặc biệt.',
]

print('\n  DEMO PREDICTION')
print('  ' + '─'*65)
for text in samples:
    r = predictor.predict(text)
    bar = '█' * int(r['confidence'] * 22)
    print(f'  [{r["predicted_label"]}] {r["predicted_name"]:<12} {bar:<22} {r["confidence"]:.0%}')
    print(f'       "{text}"')
    print()

In [ ]:
# Lưu lần cuối
save_to_drive()
print('\n✅ Hoàn thành! Tất cả kết quả đã lưu lên Google Drive.')
print(f'   Đường dẫn: {DRIVE_DIR}')